# Assignment: Logistic Regression on Imbalanced E-Commerce Purchase Data

## Objective

An e-commerce company wants to predict whether a visitor session will lead to a purchase.

The target column is `Revenue`:

- `True`: visitor purchased
- `False`: visitor did not purchase

This is a **binary classification problem** with class imbalance. Your goal is to train logistic regression models and compare different methods for handling imbalance.

## Business Context

If the model predicts that a visitor is likely to purchase, the company may show a discount popup, a special offer, or live sales assistance.

But there is a tradeoff:

- High **recall** means we catch more actual buyers, but may show offers to many non-buyers.
- High **precision** means fewer wasted offers, but we may miss some actual buyers.

Your final answer should recommend a model based on this business tradeoff.

## Dataset

We will use the Online Shoppers Purchasing Intention dataset.

Dataset URL:

```text
https://raw.githubusercontent.com/sharmaroshan/Online-Shoppers-Purchasing-Intention/master/online_shoppers_intention.csv
```

The dataset has website session-level features such as page visits, page durations, bounce rates, exit rates, visitor type, month, weekend flag, and whether the session generated revenue.

## Tasks

Complete the notebook by filling the `TODO` sections.

You need to compare four approaches:

1. Baseline Logistic Regression with default threshold `0.5`
2. Threshold tuning using predicted probabilities
3. Logistic Regression with `class_weight="balanced"`
4. SMOTE + Logistic Regression

For each approach, report:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Finally, answer the business questions at the end.

## 1. Setup

Run the install cell only if any package is missing.

In [ ]:
# Run this only if required.
# %pip install numpy pandas matplotlib seaborn scikit-learn imbalanced-learn

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the Dataset

Load the dataset from the given URL.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/sharmaroshan/Online-Shoppers-Purchasing-Intention/master/online_shoppers_intention.csv"

# TODO: Load the dataset into a DataFrame named df.
# Hint: use pd.read_csv(DATA_URL)

raise NotImplementedError("Load the dataset")

In [ ]:
# TODO: Display the first 5 rows and check the shape.

raise NotImplementedError("Inspect the dataset")

## 3. Check Class Imbalance

The target column is `Revenue`.

Check how many sessions led to purchase and how many did not.

In [ ]:
# TODO: Show count and percentage distribution of df["Revenue"].
# Hint: use value_counts() and value_counts(normalize=True)

raise NotImplementedError("Check class imbalance")

### Question 1

Is this dataset balanced or imbalanced? Why?

Write your answer here:

-

## 4. Select Features

To keep the assignment lightweight, we will use a limited set of useful features.

Numerical features:

```python
[
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]
```

Categorical features:

```python
["Month", "VisitorType", "Weekend"]
```

Target:

```python
"Revenue"
```

In [ ]:
numeric_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

categorical_features = ["Month", "VisitorType", "Weekend"]
target = "Revenue"

# TODO: Create X and y.
# X should contain numeric_features + categorical_features.
# y should be 1 for purchase and 0 for no purchase.
# Hint: df[target].astype(int)

raise NotImplementedError("Create X and y")

## 5. Train-Test Split

Use a stratified split so the train and test sets preserve the same class ratio.

In [ ]:
# TODO: Split the data into train and test sets.
# Use test_size=0.2, random_state=RANDOM_STATE, and stratify=y.

raise NotImplementedError("Create stratified train-test split")

In [ ]:
# TODO: Check class distribution in y_train and y_test.

raise NotImplementedError("Check train-test class distribution")

### Question 2

Why did we use `stratify=y` during train-test split?

Write your answer here:

-

## 6. Preprocessing

We will:

- Scale numeric features using `StandardScaler`
- One-hot encode categorical features using `OneHotEncoder`

The preprocessing should be fit only on the training data.

In [ ]:
# TODO: Create preprocessing steps for numeric and categorical features.
# Hint:
# - Use StandardScaler() for numeric features.
# - Use OneHotEncoder(handle_unknown="ignore") for categorical features.
# - Combine both using ColumnTransformer.
# - Name the numeric transformer "num" and categorical transformer "cat".
#
# For OneHotEncoder, newer scikit-learn versions use sparse_output=False,
# while older versions use sparse=False. You can use the try-except pattern below.

try:
    encoder = None  # TODO: replace None with OneHotEncoder(...)
except TypeError:
    encoder = None  # TODO: replace None with OneHotEncoder(...)

preprocessor = None  # TODO: replace None with ColumnTransformer(...)

## 7. Helper Function for Evaluation

Use this helper function to evaluate every model consistently.

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name):
    """Return a dictionary of key classification metrics."""
    # TODO: Return a dictionary with the following keys:
    # "model", "accuracy", "precision", "recall", "f1"
    #
    # Hint:
    # - "model" should store model_name.
    # - Use accuracy_score(y_true, y_pred).
    # - Use precision_score(y_true, y_pred, zero_division=0).
    # - Use recall_score(y_true, y_pred, zero_division=0).
    # - Use f1_score(y_true, y_pred, zero_division=0).
    raise NotImplementedError("Complete evaluate_predictions function")


def show_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["No Purchase", "Purchase"],
        yticklabels=["No Purchase", "Purchase"]
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 8. Baseline Logistic Regression

Train a normal logistic regression model.

Use default threshold `0.5`.

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# TODO: Fit baseline_model on X_train and y_train.
# TODO: Predict on X_test and store predictions in y_pred_baseline.

raise NotImplementedError("Train and predict baseline logistic regression")

In [ ]:
# TODO: Evaluate baseline model using evaluate_predictions.
# TODO: Show confusion matrix.
# TODO: Print classification_report.

results = []

raise NotImplementedError("Evaluate baseline model")

## 9. Threshold Tuning

Logistic regression gives probabilities:

\[
P(y=1|x)
\]

The default threshold is usually:

\[
\text{predict purchase if } P(y=1|x) \geq 0.5
\]

But in business problems, we may change the threshold depending on whether we care more about precision or recall.

Try thresholds:

```python
[0.20, 0.30, 0.40, 0.50, 0.60]
```

In [ ]:
# TODO: Get predicted probabilities for the positive class using baseline_model.predict_proba(X_test)[:, 1].
# TODO: For each threshold, convert probabilities into class predictions.
# TODO: Create a threshold comparison table with accuracy, precision, recall, and F1.

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

raise NotImplementedError("Perform threshold tuning")

In [ ]:
# TODO: Choose one threshold based on your business reasoning.
# Store it in selected_threshold.
# Example: selected_threshold = 0.30

selected_threshold = None

# TODO: Create y_pred_threshold using selected_threshold.
# TODO: Evaluate and append result to results.
# TODO: Show confusion matrix.

raise NotImplementedError("Evaluate selected threshold")

### Question 3

What happens to precision and recall when you reduce the threshold?

Why does this happen?

Write your answer here:

-

## 10. Class-Weighted Logistic Regression

Train logistic regression with:

```python
class_weight="balanced"
```

This gives more importance to the minority class during training.

In [ ]:
weighted_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

# TODO: Fit weighted_model on X_train and y_train.
# TODO: Predict on X_test and store predictions in y_pred_weighted.
# TODO: Evaluate and append result to results.
# TODO: Show confusion matrix.

raise NotImplementedError("Train and evaluate class-weighted logistic regression")

### Question 4

How did `class_weight="balanced"` change recall and precision compared to the baseline model?

Write your answer here:

-

## 11. SMOTE + Logistic Regression

SMOTE creates synthetic minority-class examples.

Important rule:

> Apply SMOTE only on the training data, never on the test data.

For this assignment, first transform the train and test data using the preprocessor, then apply SMOTE only on the transformed training data.

In [ ]:
# TODO: Fit the preprocessor on the training features.
# TODO: Use the fitted preprocessor to transform both training and test features.
#
# Hint:
# - The training set should be used to learn scaling and encoding rules.
# - The test set should only be transformed using those already-learned rules.
# - You will need one processed training feature matrix and one processed test feature matrix.

raise NotImplementedError("Preprocess train and test data for SMOTE")

In [ ]:
# TODO: Apply SMOTE only on the processed training data and y_train.
#
# Hint:
# - Create a SMOTE object with RANDOM_STATE.
# - Use the SMOTE resampling method on training data only.
# - Store the resampled features and resampled target separately.
# - Do not apply SMOTE on the test data.

raise NotImplementedError("Apply SMOTE on training data")

In [ ]:
# TODO: Train LogisticRegression on X_train_smote and y_train_smote.
# TODO: Predict on X_test_processed.
# TODO: Evaluate and append result to results.
# TODO: Show confusion matrix.

raise NotImplementedError("Train and evaluate SMOTE logistic regression")

### Question 5

Why should SMOTE be applied only on the training data?

Write your answer here:

-

## 12. Final Comparison

Create a final comparison table for all methods:

1. Baseline Logistic Regression
2. Threshold-Tuned Logistic Regression
3. Class-Weighted Logistic Regression
4. SMOTE + Logistic Regression

In [ ]:
# TODO: Convert results into a DataFrame and sort/format it clearly.

raise NotImplementedError("Create final comparison table")

## 13. Business Decision Questions

Answer these briefly.

### Question 6

If discount cost is high and the company wants to avoid showing offers to too many non-buyers, should it prioritize precision or recall?

Answer:

-

### Question 7

If missing a potential buyer is very costly, should the company prioritize precision or recall?

Answer:

-

### Question 8

Which model/method would you recommend for this business problem?

Mention:

- The model/method you selected
- The metric you prioritized
- Why this decision makes business sense

Answer:

-

## Submission Checklist

Submit your completed notebook with:

- All code cells executed
- Final comparison table
- Confusion matrices for each method
- Answers to all business questions
- A short final recommendation